In [ ]:
!nvidia-smi

Thu Feb 26 03:55:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install transformers torchaudio soundfile

In [ ]:
!pip install --upgrade sympy

In [ ]:
# dataset/LA_train/LA_train.zip
!cp /content/drive/MyDrive/dataset/LA_train/LA_train.zip /content/
!unzip -q /content/LA_train.zip -d /content/LA_train_data/
# -q là chế độ quiet, giúp Colab không in ra hàng chục ngàn dòng giải nén gây đơ trình duyệt

In [ ]:
# Cập nhật đường dẫn tương ứng với thư mục bạn vừa giải nén
PROTOCOL_PATH = "/content/LA_train_data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
AUDIO_DIR = "/content/LA_train_data/LA/ASVspoof2019_LA_train/flac/"

In [ ]:
import os
import torch
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ASVspoof2019LA(Dataset):
    def __init__(self, protocol_file, audio_dir, max_frames=64000, target_sr=16000):
        self.audio_dir = audio_dir
        self.max_frames = max_frames
        self.target_sr = target_sr

        # Parse file protocol
        self.data_list = []
        with open(protocol_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    audio_id = parts[1]
                    label_str = parts[4]
                    label = 0 if label_str == 'bonafide' else 1
                    self.data_list.append((audio_id, label))

    def __len__(self):
        return len(self.data_list)

    def process_audio(self, waveform, sr):
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform = resampler(waveform)

        num_frames = waveform.shape[1]

        if num_frames > self.max_frames:
            waveform = waveform[:, :self.max_frames]
        elif num_frames < self.max_frames:
            pad_length = self.max_frames - num_frames
            waveform = F.pad(waveform, (0, pad_length))

        return waveform

    def __getitem__(self, idx):
        audio_id, label = self.data_list[idx]
        file_path = os.path.join(self.audio_dir, f"{audio_id}.flac")

        waveform, sample_rate = torchaudio.load(file_path)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        waveform = self.process_audio(waveform, sample_rate)
        label = torch.tensor(label, dtype=torch.long)
        waveform = waveform.squeeze(0)

        return waveform, label

In [ ]:
import torch
import torch.nn as nn
import sympy
import sympy.printing
from transformers import WavLMModel

class WavLMTeacher(nn.Module):
    def __init__(self, model_name="microsoft/wavlm-base-plus", num_classes=2):
        super().__init__()
        # 1. Khởi tạo WavLM Pre-trained từ Hugging Face
        self.wavlm = WavLMModel.from_pretrained(model_name)

        # Lấy kích thước hidden dimension của model (thường là 768 cho bản base)
        hidden_size = self.wavlm.config.hidden_size

        # 2. Xây dựng Anti-spoof Head (Mạng MLP nhỏ) để phân loại
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, waveform):
        """
        waveform: Tensor kích thước (batch_size, sequence_length)
        """
        # Trích xuất đặc trưng từ WavLM
        outputs = self.wavlm(waveform)

        # Lấy hidden states của lớp cuối cùng: (Batch, Frames, 768)
        hidden_states = outputs.last_hidden_state

        # 1. Mean Pooling: Tính trung bình theo chiều thời gian (dim=1) -> (Batch, 768)
        pooled_features = hidden_states.mean(dim=1)

        # 2. Đưa qua Classification Head -> (Batch, 2)
        logits = self.head(pooled_features)

        # Trả về cả logit (để tính Loss phân loại) và hidden_states (để dùng cho Distillation sau này)
        return logits, hidden_states

In [ ]:
# Khởi tạo Dataset và DataLoader
train_dataset = ASVspoof2019LA(PROTOCOL_PATH, AUDIO_DIR)

# LƯU Ý: Tùy vào VRAM của GPU (T4 thường có 16GB VRAM), bạn hãy chỉnh batch_size cho phù hợp.
# Nếu bị lỗi Out of Memory (OOM), hãy giảm batch_size xuống 8 hoặc 4.
train_loader = DataLoader(
    train_dataset,
    batch_size=64,     # (1) Nâng từ 12 lên 64 (hoặc 128 nếu dùng A100). Giúp train nhanh hơn rất nhiều.
    shuffle=True,
    num_workers=4,     # (2) Nâng từ 2 lên 4 hoặc 8. Tận dụng CPU mạnh của bản Pro để đọc data song song.
    pin_memory=True    # (3) Bắt buộc bật: Tăng tốc độ bơm dữ liệu từ RAM sang VRAM của GPU.
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WavLMTeacher().to(device)

# Áp dụng chiến lược đóng băng (Freezing Strategy)
model.wavlm.feature_extractor._freeze_parameters()
for i in range(6):
    for param in model.wavlm.encoder.layers[i].parameters():
        param.requires_grad = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

In [ ]:
####### Mới


import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast # Import công cụ AMP

# Cấu hình Optimizer như cũ
wavlm_trainable_params = [p for p in model.wavlm.encoder.layers[6:].parameters() if p.requires_grad]
head_params = [p for p in model.head.parameters() if p.requires_grad]

optimizer = optim.AdamW([
    {'params': wavlm_trainable_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-4}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()
epochs = 10

# Khởi tạo GradScaler để scale gradient khi dùng FP16 (tránh tràn số)
scaler = GradScaler()

print(f"Bắt đầu Fine-tune Giáo viên với AMP trên thiết bị: {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

    for batch_idx, (waveforms, labels) in enumerate(progress_bar):
        waveforms = waveforms.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # BẬT AUTOCAST: Ép PyTorch tự động tính toán bằng FP16 ở các layer phù hợp
        with torch.amp.autocast('cuda'):
            logits, _ = model(waveforms)
            loss = criterion(logits, labels)

        # BACKWARD VỚI SCALER
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Thống kê kết quả
        running_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        current_loss = running_loss / (batch_idx + 1)
        current_acc = 100 * correct / total

        progress_bar.set_postfix({
            'Loss': f'{current_loss:.4f}',
            'Acc': f'{current_acc:.2f}%'
        })

    print(f"✅ Hoàn thành Epoch {epoch+1} | Average Loss: {running_loss/len(train_loader):.4f} | Final Acc: {100 * correct / total:.2f}%")
    print("-" * 60)

torch.save(model.state_dict(), "wavlm_teacher_finetuned.pth")
print("🎉 Đã lưu trọng số mô hình Giáo viên thành công!")

/tmp/ipython-input-6749/3058756907.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Bắt đầu Fine-tune Giáo viên với AMP trên thiết bị: cuda...


Epoch 1/10:   0%|          | 0/397 [00:00<?, ?it/s]/tmp/ipython-input-6749/3058756907.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/10: 100%|██████████| 397/397 [00:58<00:00,  6.78it/s, Loss=0.1538, Acc=95.09%]


✅ Hoàn thành Epoch 1 | Average Loss: 0.1538 | Final Acc: 95.09%
------------------------------------------------------------


Epoch 2/10: 100%|██████████| 397/397 [00:55<00:00,  7.17it/s, Loss=0.0281, Acc=99.02%]


...

✅ Hoàn thành Epoch 8 | Average Loss: 0.0124 | Final Acc: 99.59%
------------------------------------------------------------


Epoch 9/10: 100%|██████████| 397/397 [00:55<00:00,  7.12it/s, Loss=0.0050, Acc=99.85%]


✅ Hoàn thành Epoch 9 | Average Loss: 0.0050 | Final Acc: 99.85%
------------------------------------------------------------


Epoch 10/10: 100%|██████████| 397/397 [00:55<00:00,  7.14it/s, Loss=0.0064, Acc=99.78%]


✅ Hoàn thành Epoch 10 | Average Loss: 0.0064 | Final Acc: 99.78%
------------------------------------------------------------
🎉 Đã lưu trọng số mô hình Giáo viên thành công!


In [ ]:
import shutil
drive_path = "/content/drive/MyDrive/wavlm_teacher_finetuned.pth"
save_path_colab = "/content/wavlm_teacher_finetuned.pth"
shutil.copy(save_path_colab, drive_path)
print(f"Đã sao chép mô hình an toàn sang Google Drive tại: {drive_path}")

Đã sao chép mô hình an toàn sang Google Drive tại: /content/drive/MyDrive/wavlm_teacher_finetuned.pth


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU DEV SET
# ==========================================
# THAY ĐỔI ĐƯỜNG DẪN NÀY CHO PHÙ HỢP VỚI THƯ MỤC BẠN ĐÃ GIẢI NÉN TRÊN COLAB
DEV_PROTOCOL_PATH = "/content/LA_train_data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"
DEV_AUDIO_DIR = "/content/LA_train_data/LA/ASVspoof2019_LA_eval/flac/"

# ==========================================
# 2. HÀM TÍNH TOÁN EER
# ==========================================
def compute_eer(y_true, y_score):
    """Tính toán Equal Error Rate (EER)"""
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    thresh = interp1d(fpr, thresholds)(eer)
    return eer, thresh

# ==========================================
# 3. KHỞI TẠO DATASET VÀ MÔ HÌNH
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

# Khởi tạo Dev Dataloader (Dùng class ASVspoof2019LA đã định nghĩa từ trước)
try:
    dev_dataset = ASVspoof2019LA(protocol_file=DEV_PROTOCOL_PATH, audio_dir=DEV_AUDIO_DIR)
    dev_loader = DataLoader(
    dev_dataset,
    batch_size=128,      # (1) Tăng lên 128 (thậm chí 256). A100 dư sức "nhai" mẻ dữ liệu lớn này.
    shuffle=False,       # Lưu ý: Lúc đánh giá thì tuyệt đối KHÔNG shuffle (trộn) dữ liệu
    num_workers=8,       # (2) Tăng lên 8. Máy chủ A100 của Colab Pro thường cấp 8-12 CPU cores.
    pin_memory=True      # (3) Giữ nguyên True để bơm data từ RAM sang VRAM nhanh nhất
)
except Exception as e:
    print(f"Lỗi load dữ liệu. Vui lòng kiểm tra lại đường dẫn Dev Set: {e}")

# Khởi tạo mô hình và LOAD TRỌNG SỐ ĐÃ TRAIN
model = WavLMTeacher().to(device)

model_path = "wavlm_teacher_finetuned.pth"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"✅ Đã load thành công trọng số từ: {model_path}")
else:
    print(f"❌ Không tìm thấy file {model_path}. Hãy đảm bảo file đang nằm trong thư mục hiện tại của Colab.")

# ==========================================
# 4. VÒNG LẶP ĐÁNH GIÁ (EVALUATION)
# ==========================================
def evaluate_model(model, dataloader, device):
    model.eval() # Bắt buộc: Chuyển sang chế độ evaluation (tắt Dropout, v.v.)

    all_scores = []
    all_labels = []

    print("\nĐang tiến hành đánh giá trên tập Validation...")
    progress_bar = tqdm(dataloader, desc="Evaluating", leave=True)

    with torch.no_grad(): # Tắt tính gradient để chạy nhanh và tiết kiệm VRAM
        with torch.amp.autocast('cuda'): # Dùng AMP để đồng bộ với lúc train
            for waveforms, labels in progress_bar:
                waveforms = waveforms.to(device)

                # Chạy qua mô hình
                logits, _ = model(waveforms)

                # Tính xác suất (Softmax)
                probs = F.softmax(logits, dim=1)

                # Lấy điểm số của class Bonafide (cột 0)
                bonafide_scores = probs[:, 0].cpu().numpy()
                all_scores.extend(bonafide_scores)

                # Chuyển đổi nhãn (Bonafide = 1, Spoof = 0 để phù hợp hàm tính EER)
                labels_np = labels.numpy()
                true_labels = 1 - labels_np
                all_labels.extend(true_labels)

    # Tính toán EER
    eer, threshold = compute_eer(all_labels, all_scores)
    print("\n" + "="*50)
    print(f"🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP DEV:")
    print(f"👉 Equal Error Rate (EER): {eer * 100:.3f}%")
    print(f"👉 Ngưỡng quyết định (Threshold) tối ưu: {threshold:.4f}")
    print("="*50)

    return eer

# Chạy đánh giá
if 'dev_loader' in locals():
    eer_result = evaluate_model(model, dev_loader, device)

Đang sử dụng thiết bị: cuda


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

✅ Đã load thành công trọng số từ: wavlm_teacher_finetuned.pth

Đang tiến hành đánh giá trên tập Validation...


Evaluating: 100%|██████████| 557/557 [01:06<00:00,  8.36it/s]


🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP DEV:
👉 Equal Error Rate (EER): 1.618%
👉 Ngưỡng quyết định (Threshold) tối ưu: 0.9489
